# core

> web search made easy

core is the fetch-and-read layer. `fetch()` handles any URL from a simple static page to a JavaScript-rendered SPA; `to_md()` strips HTML to clean markdown. The module also covers pagination across JSON APIs, reading arXiv papers and YouTube transcripts, and cloning GitHub repos.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from fastcore.all import Path, L, timed_cache, globtastic, startthread, parallel, joins, run, first
from fastcore.xdg import xdg_cache_home
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from diskcache import Index
import re, httpx, shutil, time
import lxml.etree as etree
from pdf_oxide import PdfDocument
from scrapling import Selector
from scrapling.fetchers import Fetcher, StealthyFetcher, DynamicFetcher
from urllib.parse import urljoin, urlparse
import yt_dlp

In [ ]:
#| export
@retry(wait=wait_exponential(1,30, min=2),stop=stop_after_attempt(4),
       retry=retry_if_exception_type((httpx.TimeoutException, IOError)))
def http_get(url, **kw):
    r = httpx.get(url, timeout=30, **kw)
    if r.status_code == 429: raise IOError(f"Rate limited (429): {url}")
    return r

@retry(wait=wait_exponential(1,30, min=2),stop=stop_after_attempt(4),
       retry=retry_if_exception_type((httpx.TimeoutException, IOError)))
def http_post(url, **kw):
    r = httpx.post(url, timeout=30, **kw)
    if r.status_code == 429: raise IOError(f"Rate limited (429): {url}")
    return r

def save_path(path=None):
    "Get cache path for `name` (e.g. 'arxiv' or 'fetch')"
    p = xdg_cache_home()/'.fossick'
    if path: p /= path
    p.parent.mkdir(parents=True, exist_ok=True)
    return p

In [ ]:
#| export
_arxin = Index(str(save_path('arxiv_cache.db')))
_fetch_cache = {}
def _fetch_arxiv_meta(arxiv_id:str):
    "Fetch and parse arxiv metadata for a given ID; result is cached to disk"
    r = http_get(f'https://export.arxiv.org/api/query?id_list={arxiv_id}')
    if r.status_code != 200: raise Exception(f"Failed to fetch arxiv data: {r.status_code}")
    ns = {'a': 'http://www.w3.org/2005/Atom'}
    entry = etree.fromstring(r.content).find('a:entry', ns)
    if entry is None: raise Exception("No paper found")
    def txt(tag): return entry.find(f'a:{tag}', ns).text
    pu = first(L(entry.findall('a:link',ns)).filter(lambda l: l.get('title')=='pdf').map(lambda l: l.get('href')))
    au = L(entry.findall('a:author', ns)).map(lambda a: a.find('a:name', ns).text)
    return dict(title=txt('title').strip(), summary=txt('summary').strip(),
                published=txt('published'), link=txt('id'), pdf_url=pu, authors=au)

In [ ]:
#| export
def get_pdf(url:str):
    "Fetch PDF from URL and return as PdfDocument"
    r = http_get(url, follow_redirects=True)
    if r.status_code != 200: raise Exception(f"Failed to fetch PDF: {r.status_code}")
    return PdfDocument.from_bytes(r.content)

In [ ]:
#| export
def read_arxiv(url:str, # arxiv PDF URL, or arxiv abstract URL, or arxiv ID
               save_pdf:bool=True, # if True, saves the downloaded PDF to disk
               save_dir:str='.', # directory in which to save the PDF
               force:bool=False # if True, forces re-download of PDF even if it exists on disk
               ):
    "Get paper information from arxiv URL or ID, optionally saving PDF to disk"
    arxiv_id = url.rsplit('/', 1)[-1]
    ver = re.search(r'v(\d+)$', arxiv_id)
    ver_sfx = f'v{ver.group(1)}' if ver else ''
    arxiv_id = re.sub(r'v\d+$', '', arxiv_id)
    pdf_path = Path(save_dir)/f'{arxiv_id}{ver_sfx}.pdf'
    if not force and arxiv_id in _arxin: return _arxin[arxiv_id]
    else: res = dict(_fetch_arxiv_meta(arxiv_id))
    if not res['pdf_url']: raise Exception("No PDF URL found in arxiv metadata")
    doc = get_pdf(res['pdf_url']) if not pdf_path.exists() else PdfDocument(str(pdf_path))
    if save_pdf and not pdf_path.exists():
        pdf_path.parent.mkdir(parents=True, exist_ok=True)
        doc.save(str(pdf_path), compress=True, garbage_collect=True)
        res |= dict(pdf_path=pdf_path)
    res |= dict(source=doc.to_markdown_all())
    _arxin[arxiv_id] = res
    return res

In [ ]:
#| export
def _gh_ssh(url:str): # GitHub URL, SSH remote, or bare `user/repo`
    'Convert GitHub URL to SSH remote; pass through if already SSH; return None if not GitHub'
    if url.startswith('git@github.com:'): return url
    if m := re.match(r'https://github\.com/([^/]+)/([^/]+)', url):
        return f'git@github.com:{m.group(1)}/{m.group(2)}.git'

def _get_git_repo(gh_ssh:str):
    'Clone or update a GitHub repo to local cache; return Path'
    repo_name = gh_ssh.rsplit('/', 1)[-1].removesuffix('.git')
    repo_dir = save_path('git_clones')/repo_name
    if repo_dir.exists():
        try:
            run(['git', '-C', str(repo_dir), 'fetch'])
            branch = run(['git', '-C', str(repo_dir), 'symbolic-ref', 'refs/remotes/origin/HEAD']).split('/')[-1]
            run(['git', '-C', str(repo_dir), 'reset', '--hard', f'origin/{branch}'])
            return repo_dir
        except IOError: shutil.rmtree(repo_dir)
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    run(['git', 'clone', gh_ssh, str(repo_dir)])
    return repo_dir

@timed_cache(seconds=3600)
def read_gh_repo(path_or_url:str,  # GitHub URL, SSH address, or local path
                 globs:tuple=None,  # file glob patterns (default: README*, pyproject.toml, *.py)
                 limit:int=None,    # max files to return
                 as_list:bool=False # return list of Paths instead of {path: content} dict
                ):
    'Read files from a GitHub repo filtered by glob patterns'
    ssh = _gh_ssh(path_or_url)
    repo_dir = _get_git_repo(ssh) if ssh else Path(path_or_url)
    if globs is None: globs = ('README*', 'pyproject.toml', '*.py')
    files = L(p for g in globs for p in globtastic(repo_dir, file_glob=g, func=Path)).unique()
    if limit: files = files[:limit]
    if as_list: return files
    return {str(p): p.read_text(errors='ignore') for p in files}

In [ ]:
#| export
def read_gh_file(url:str # GitHub blob URL of the file to read
                ):
    'Read raw contents of a file from its GitHub URL'
    raw_url = re.sub(r'https://github\.com/([^/]+)/([^/]+)/blob/([^/]+)/(.+)',
                     r'https://raw.githubusercontent.com/\1/\2/refs/heads/\3/\4', url)
    if (r:=httpx.get(raw_url)).status_code != 200: raise Exception(f"Failed to fetch {raw_url}: {r.status_code}")
    return r.text

In [ ]:
read_arxiv('https://arxiv.org/abs/2306.14881')['summary'][:200]

'Low-metallicity dwarf galaxies often show no or little CO emission, despite the intense star formation observed in local samples. Both simulations and resolved observations indicate that molecular gas'

In [ ]:
read_gh_file('https://github.com/Karthik777/litesearch/blob/main/README.md')[:200]

'# litesearch\n\n\n<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->\n\n> **NB** Reading this on GitHub? The formatted\n> [documentation](https://Karthik777.github.io/litesearch/) is nicer.\n\nlitese'

In [ ]:
#| eval: false
list(read_gh_repo('https://github.com/vedicreader/gheasy'))

['/Users/71293/.cache/.fossick/git_clones/gheasy/README.md',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/pyproject.toml',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/__init__.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/_modidx.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/core.py',
 '/Users/71293/.cache/.fossick/git_clones/gheasy/gheasy/workflow.py']

## Web Fetching

`fetch()` returns a dict with `url`, `status`, `html`, `data` (parsed JSON when the response is JSON), and `xhr` (captured network calls when `capture_xhr=True`). `to_md()` produces clean markdown, optionally extracting just the element matched by a CSS selector.

In [ ]:
#| export
def _html(x): return x['html'] if isinstance(x, dict) else x
def _decode(b): return b.decode('utf-8', errors='replace') if isinstance(b, bytes) else b
def _wrap_md(text, tag): return f'\n<{tag}>\n{text}\n</{tag}>\n' if text else ''

def clean_md(text, rm_comments=True, rm_details=True):
    "Remove comments and `<details>` sections from `text`"
    if rm_comments: text = re.sub(r'\n?<!--.*?-->\n?', '', text, flags=re.DOTALL)
    if rm_details: text = re.sub(r'\n?<details>.*?</details>\n?', '', text, flags=re.DOTALL)
    return text

def html2md(s:str, ignore_links=True):
    "Convert `s` from HTML to markdown"
    import html2text
    o = html2text.HTML2Text(bodywidth=5000)
    o.ignore_links, o.mark_code, o.ignore_images = ignore_links, True, True
    return o.handle(s)

def to_md(page_or_html, # Page dict (from fetch/crawl) or raw HTML string
          sel:str=None, # CSS selector to extract before conversion; returns '' if no match
          multi:bool=False, # Return all selector matches joined
          wrap_tag:str=None, # Wrap each multi-result in <wrap_tag>...</wrap_tag>; only used when multi=True
          ignore_links:bool=True,
          rm_comments:bool=True,
          rm_details:bool=True,
         ) -> str:
    "Convert a Page dict or HTML string to clean markdown"
    html = _html(page_or_html)
    _md = lambda h: clean_md(html2md(str(h), ignore_links), rm_comments, rm_details)
    if sel:
	    els = Selector(html).css(sel)
	    mds = L(els).map(lambda m: _md(m.get()))
	    if multi: return joins('\n',mds.map(lambda m: _wrap_md(m, wrap_tag) if wrap_tag else m))
	    html = str(mds[0]) if mds else ''
    return _md(html)

In [ ]:
#| export
def _get_page(url, method='GET', payload=None, heavy=False, stealthy=False, **kw):
    if method.upper() == 'POST': return Fetcher.post(url, json=payload, **kw)
    if payload: kw['params'] = payload
    if not (heavy or stealthy): return Fetcher.get(url, **kw)
    f = DynamicFetcher.fetch if heavy else StealthyFetcher.fetch
    t = startthread(lambda: f(url, **kw)); t.join(); return t.result

In [ ]:
#| export
def fetch(url:str,                # URL to fetch
          sel:str=None,           # CSS selector to extract (None = full page)
          method:str='GET',       # HTTP method; 'POST' sends payload as JSON body
          payload:dict=None,      # POST body (JSON) or GET query params
          heavy:bool=False,       # Full JS rendering via headless browser
          stealthy:bool=False,    # Anti-bot stealth fetcher (Cloudflare etc.)
          capture_xhr:bool=False, # Intercept XHR/fetch calls; forces heavy=True
          cache:bool=False,       # Cache successful responses to disk by URL+sel
          force:bool=False,       # If True, forces re-fetch even if cached result exists
          **kw,                   # Extra kwargs passed to scrapling (e.g. verify, headers)
         ) -> dict:
    "Fetch `url`, return Page dict {url, status, html, data, xhr} where html is raw response body"
    _key = f"{url}\x00{sel or ''}"
    if force and _key in _fetch_cache: del _fetch_cache[_key]
    if cache and _key in _fetch_cache: return _fetch_cache[_key]
    heavy = heavy or capture_xhr
    if capture_xhr: kw.setdefault('capture_xhr', '.*')
    page = _get_page(url, method=method, payload=payload, heavy=heavy, stealthy=stealthy, **kw)
    html = body = _decode(page.body)
    if sel: html = str(el.get()) if (el:=page.css(sel).first) else ''
    try: data = json.loads(body)
    except (ValueError, TypeError): data = None
    except Exception as ex: data = f"Error parsing JSON: {ex}"
    xhr = []
    if capture_xhr:
        for x in page.captured_xhr:
            ct = next((v for k,v in dict(x.headers).items() if 'content-type' in k.lower()), '')
            b = _decode(x.body)
            try: d = json.loads(b)
            except (ValueError, TypeError): d = b
            xhr.append({'url': x.url, 'content_type': ct, 'data': d})
    result = {'url': url, 'status': page.status, 'html': html, 'data': data, 'xhr': xhr}
    if cache and result['status'] == 200: _fetch_cache[_key] = result
    return result

In [ ]:
#| export
def crawl(start_url:str,               # URL to start from
          sel:str=None,                # CSS selector to extract per page
          follow_sel:str='a[href]',    # CSS selector for links to follow
          same_domain:bool=True,       # Only follow links on same domain
          max_pages:int=10,            # Max pages to visit
          delay:float=0,               # Seconds to wait between requests (polite crawling)
          heavy:bool=False,
          stealthy:bool=False,
          **kw,                        # Extra kwargs passed to scrapling (e.g. verify, timeout)
         ) -> list:
    "Crawl from `start_url`, following `follow_sel` links, return list of Page dicts"
    bd = urlparse(start_url).netloc
    v,q,res = set(),[start_url],[]
    while q and len(v) < max_pages:
        url = q.pop(0)
        if url in v: continue
        v.add(url)
        if delay and res: time.sleep(delay)
        try:
            pg = fetch(url, heavy=heavy, stealthy=stealthy, **kw)
            if pg['status'] != 200: continue
            page = Selector(pg['html'])
            if sel: pg['html'] = str(el.get()) if (el:= page.css(sel).first) else ''
            res.append(pg)
            for link in page.css(follow_sel):
                href = link.attrib.get('href', '')
                if not href or href.startswith(('#', 'javascript:', 'mailto:')): continue
                abs_url = urljoin(url, href)
                if same_domain and urlparse(abs_url).netloc != bd: continue
                if abs_url not in v: q.append(abs_url)
        except Exception as ex:
	        print(f"Error fetching {url}: {ex}. Continuing crawl.")
	        continue
    return res

In [ ]:
#| export
def get_options(page_or_html,  # Page dict (from fetch) or raw HTML string
                sel:str         # CSS selector for the <select> element
               ) -> list:
    "Extract options from a <select> element; returns [{'value': ..., 'text': ...}]"
    return [{'value': el.attrib.get('value', el.text or ''), 'text': (el.text or '').strip()}
            for el in Selector(_html(page_or_html)).css(f'{sel} option')]

def fetch_all(urls:list,           # URLs to fetch
              sel:str=None,         # CSS selector to extract per page (None = full page)
              concurrency:int=8,    # Max parallel fetches
              heavy:bool=False,
              stealthy:bool=False,
              **kw                  # Extra kwargs passed to fetch()
             ) -> list:
    "Fetch a list of URLs in parallel; returns Page dicts in the same order as urls"
    return parallel(fetch, urls, sel=sel, heavy=heavy, stealthy=stealthy, threadpool=True, n_workers=concurrency, **kw)

In [ ]:
_sel_html = '''<html><body>
<select id="kanda">
  <option value="1">Balakanda</option>
  <option value="2">Ayodhyakanda</option>
  <option value="3">Aranyakanda</option>
</select>
</body></html>'''

opts = get_options(_sel_html, '#kanda')
assert opts == [{'value': '1', 'text': 'Balakanda'},
                {'value': '2', 'text': 'Ayodhyakanda'},
                {'value': '3', 'text': 'Aranyakanda'}]

# accepts Page dict
_page = {'url': 'x', 'status': 200, 'html': _sel_html, 'data': None, 'xhr': []}
assert get_options(_page, '#kanda') == opts

# no match → empty list
assert get_options(_sel_html, '#missing') == []

# fetch_all: parallel fetch, order preserved
_urls = ['https://httpbin.org/get', 'https://httpbin.org/ip']
_pages = fetch_all(_urls, verify=False)
assert len(_pages) == 2
assert _pages[0]['url'] == _urls[0]
assert _pages[1]['url'] == _urls[1]
assert all(p['status'] == 200 for p in _pages)

# live: Valmiki Ramayana — discover sargas 1–3 of Balakanda and Ayodhyakanda, fetch in parallel
_base = 'https://www.valmiki.iitk.ac.in/sloka'
_home = fetch(f'{_base}?field_kanda_tid=1&language=dv&field_sarga_value=1')
_kandas = [o for o in get_options(_home, '#edit-field-kanda-tid') if o['value']]  # drop placeholder
assert len(_kandas) >= 6, f"Expected ≥6 kandas, got {len(_kandas)}: {_kandas}"
assert any('BALA' in k['text'].upper() for k in _kandas)

for _k in _kandas[:2]:  # Balakanda, Ayodhyakanda
    _kp = fetch(f'{_base}?field_kanda_tid={_k["value"]}&language=dv&field_sarga_value=1')
    _sargas = [o for o in get_options(_kp, '#edit-field-sarga-value') if o['value']]
    assert len(_sargas) > 0, f"No sargas found for {_k['text']}"
    _urls = [f'{_base}?field_kanda_tid={_k["value"]}&language=dv&field_sarga_value={s["value"]}'
             for s in _sargas[:3]]
    _pages = fetch_all(_urls, sel='.view-content')
    assert len(_pages) == 3
    assert all(p['status'] == 200 for p in _pages)
    assert all(len(to_md(p)) > 50 for p in _pages), "Expected non-trivial markdown content"
    print(f"{_k['text']}: {len(_sargas)} sargas, first 3 fetched OK")
    print(f"  sarga 1 preview: {to_md(_pages[0])[:120]!r}")

[2026-06-03 08:26:24] INFO: Fetched (200) <GET https://httpbin.org/get> (referer: https://www.google.com/)
[2026-06-03 08:26:24] INFO: Fetched (200) <GET https://httpbin.org/ip> (referer: https://www.google.com/)
[2026-06-03 08:26:26] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=1> (referer: https://www.google.com/)
[2026-06-03 08:26:26] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=1> (referer: https://www.google.com/)
[2026-06-03 08:26:27] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=1> (referer: https://www.google.com/)
[2026-06-03 08:26:28] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&language=dv&field_sarga_value=2> (referer: https://www.google.com/)
[2026-06-03 08:26:28] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=1&langu

BALAKANDA: 77 sargas, first 3 fetched OK
  sarga 1 preview: '[Saint Narada visits hermitage of Valmiki -- Valmiki queries about a single perfect individual bestowed with all good qu'


[2026-06-03 08:26:29] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=2&language=dv&field_sarga_value=1> (referer: https://www.google.com/)
[2026-06-03 08:26:30] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=2&language=dv&field_sarga_value=1> (referer: https://www.google.com/)
[2026-06-03 08:26:30] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=2&language=dv&field_sarga_value=3> (referer: https://www.google.com/)
[2026-06-03 08:26:30] INFO: Fetched (200) <GET https://www.valmiki.iitk.ac.in/sloka?field_kanda_tid=2&language=dv&field_sarga_value=2> (referer: https://www.google.com/)


AYODHYAKANDA: 119 sargas, first 3 fetched OK
  sarga 1 preview: "[Description of Rama's virtues Dasaratha contemplates to install Rama as heirapparent Invites kings and elders from town"


## API Discovery & Pagination

`find_xhr()` visits a page with a real browser, captures all XHR and fetch calls it makes, and returns those matching a URL pattern. This surfaces the undocumented JSON endpoints that JavaScript-heavy sites use to load their data. `paginate_api()` replays one of those captured requests across pages until results are exhausted.

In [ ]:
#| export
def compile_pattern(pattern):
	"Compile pattern as regex; if invalid (e.g. bare glob like *foo*), convert via fnmatch first"
	try: return re.compile(pattern)
	except re.error:
		import fnmatch
		return re.compile(fnmatch.translate(pattern))

def find_xhr(url:str,             # URL to visit with browser
             pattern:str='*',     # Glob or regex pattern to filter captured XHR URLs
             json_only:bool=True, # Return only JSON responses
             **kw,                # Extra kwargs passed to fetch (verify, network_idle, etc.)
            ) -> list:
    "Visit `url` with a headless browser, return [{url, content_type, data}] for each XHR/fetch call made"
    kw.setdefault('network_idle', True)
    rx = compile_pattern(pattern)
    xhr = fetch(url, capture_xhr=True, **kw)['xhr']
    filtered = [x for x in xhr if rx.search(x['url'])]
    if json_only: return [x for x in filtered if 'json' in x['content_type']]
    return filtered

In [ ]:
assert compile_pattern('.*products.*').search('https://api.example.com/products?q=1')
assert compile_pattern('*products*').search('https://api.example.com/products?q=1')
assert not compile_pattern('*products*').search('https://api.example.com/search')
assert compile_pattern('.*[Ss]earch.*').search('https://api.example.com/Search')

In [ ]:
#| export
def paginate_api(url:str,                      # API endpoint URL
                 payload:dict=None,            # Request body (POST) or params (GET)
                 page_field:str='pageNumber',  # Payload key to increment for each page
                 size_field:str='pageSize',    # Payload key for page size (detects last page)
                 results_field:str=None,       # Response key with items list (auto-detect if None)
                 method:str='POST',            # HTTP method
                 max_pages:int=10,
                 page_size=24,                 # Page size to request (only used if not in payload)
                 page_start:int=1,             # Starting page number (default 1)
                 save:bool=False,              # If True, saves each page's items to disk
                 save_file:str='{url}_page_{page}.json', # Filepath pattern for saving (only used if save=True)
                 force:bool=False,             # If True, forces re-fetching even if saved file exists
                 **kw,                         # Extra kwargs passed to fetch() (verify, headers, etc.)
                ) -> list:
    "Paginate through a JSON API, collecting all results. Auto-detects the items list in response."
    payload = dict(payload or {})
    page_size = payload.get(size_field, page_size)
    all_items = []
    for page_num in range(page_start, max_pages + page_start):
        sf = save_file.format(url=urlparse(url).netloc, page=page_num)
        if not force and Path(sf).exists():
            print(f"Page {page_num} already saved, skipping fetch")
            with open(sf) as f: all_items.extend(json.load(f)); continue
        pg = fetch(url, method=method, payload={**payload, page_field: page_num}, **kw)
        if pg['status'] != 200: break
        data = pg['data']
        if data is None: break
        if isinstance(data, list): items = data
        elif results_field:
            raw = data.get(results_field, [])
            items = list(raw.values()) if isinstance(raw, dict) else raw
        else:
            lists = {k: v for k, v in data.items() if isinstance(v, list)}
            items = max(lists.values(), key=len) if lists else []
        if not items: break
        all_items.extend(items)
        if len(items) < page_size: break
        if save: Path(sf).write_text(json.dumps(items, indent=2))
    return all_items

In [ ]:
from fastcore.test import test_eq

In [ ]:
test_eq(clean_md('before <!-- a comment --> after'), 'before  after')
# surrounding newlines are consumed along with the block
test_eq(clean_md('a\n<details>hidden</details>\nb'), 'ab')
test_eq(clean_md('a\n\n<details>hidden</details>\n\nb'), 'a\n\nb')
test_eq(clean_md('no change', rm_comments=False, rm_details=False), 'no change')
md = html2md('<h1>Hello</h1><p>World</p>')
assert '# Hello' in md and 'World' in md

# to_md tests
html_ = '<html><body><h1>Hello</h1><p>World</p><p class="x">Keep</p></body></html>'
_md = to_md(html_)
assert 'Hello' in _md
assert 'World' in _md

# accepts Page dict — extracts html field
_page = {'url': 'https://example.com', 'status': 200, 'html': html_, 'data': None, 'xhr': []}
assert to_md(_page) == _md

# sel extracts first matching element only
_md_h1 = to_md(html_, sel='h1')
assert 'Hello' in _md_h1
assert 'World' not in _md_h1

# multi=True returns all matches joined
_md_ps = to_md(html_, sel='p', multi=True)
assert 'World' in _md_ps
assert 'Keep' in _md_ps

# sel with no match returns empty string
_md_none = to_md(html_, sel='div.missing')
assert _md_none == '' or len(_md_none) < 5  # html2md of empty string may produce minimal whitespace

# wrap_tag wraps each multi-result
_md_wrapped = to_md(html_, sel='p', multi=True, wrap_tag='item')
assert '<item>' in _md_wrapped
assert '</item>' in _md_wrapped

In [ ]:
_pg = fetch('https://httpbin.org/get', verify=False)
assert isinstance(_pg, dict), f"Expected dict, got {type(_pg)}"
assert set(_pg.keys()) == {'url', 'status', 'html', 'data', 'xhr'}, f"Keys mismatch: {_pg.keys()}"
assert _pg['status'] == 200, f"Expected 200, got {_pg['status']}"
assert _pg['xhr'] == [], "xhr should be empty without capture_xhr"
assert len(_pg['html']) > 0, "html should be non-empty"

# httpbin.org/get returns JSON — data should be parsed dict
assert _pg['data'] is not None, "data should be parsed JSON for a JSON response"
assert _pg['data']['url'] == 'https://httpbin.org/get'

# to_md integration — fetch + convert
_text = to_md(_pg)
assert isinstance(_text, str) and len(_text) > 0

[2026-06-03 08:26:53] INFO: Fetched (200) <GET https://httpbin.org/get> (referer: https://www.google.com/)


In [ ]:
_pages = crawl('https://httpbin.org', max_pages=2, verify=False)
assert isinstance(_pages, list), f"Expected list, got {type(_pages)}"
assert len(_pages) > 0, "Expected at least one page"
assert all(isinstance(p, dict) for p in _pages)
assert all(set(p.keys()) == {'url', 'status', 'html', 'data', 'xhr'} for p in _pages), \
    f"Unexpected keys: {[set(p.keys()) for p in _pages]}"
assert all(p['status'] == 200 for p in _pages), "Non-200 pages should be skipped"
assert all(len(p['html']) > 0 for p in _pages), "html should be non-empty"
assert len({p['url'] for p in _pages}) == len(_pages), "url values should be unique"

[2026-06-03 08:27:03] INFO: Fetched (200) <GET https://httpbin.org/> (referer: https://www.google.com/)
[2026-06-03 08:27:03] INFO: Fetched (200) <GET https://httpbin.org/forms/post> (referer: https://www.google.com/)


In [ ]:
#| eval: false
# Step 1: visit the listing page with a headless browser, capture all XHR/fetch calls
apis = find_xhr('https://www.danmurphys.com.au/list/wine-all', verify=False)

[2026-06-03 08:27:22] INFO: Fetched (200) <GET https://www.danmurphys.com.au/list/wine-all> (referer: https://www.google.com/)
[2026-06-03 08:27:22] INFO: Fetched (200) <GET https://aem.danmurphys.com.au/list/wine-all.model.json> (referer: https://www.danmurphys.com.au/)
[2026-06-03 08:27:22] INFO: Fetched (200) <GET https://aem.danmurphys.com.au/content/experience-fragments/dm/au/en/site/gsa/global-alert/master.model.json> (referer: https://www.danmurphys.com.au/)
[2026-06-03 08:27:22] INFO: Fetched (200) <GET https://aem.danmurphys.com.au/content/experience-fragments/dm/au/en/site/header/meganav-updated-2026/meganav/master2.model.json> (referer: https://www.danmurphys.com.au/)
[2026-06-03 08:27:22] INFO: Fetched (404) <GET https://aem.danmurphys.com.au/content/experience-fragments/dm/au/en/site/hello_bar/hello-bar/master.model.json> (referer: https://www.danmurphys.com.au/)
[2026-06-03 08:27:22] INFO: Fetched (200) <GET https://www.danmurphys.com.au/dd-web-assets/address/allcitiesadd

In [ ]:
#| eval: false
# Dan Murphy's wines — full workflow: discover API → paginate → save JSON
#
# Dan Murphy's is a SPA: the product listing page loads products via a hidden JSON API.
# Step 1 visits with a real browser to intercept those calls; step 2 replays the API
# directly (no browser needed) to collect all 120 wines with full pricing data.
# find the product API — large JSON response containing 'Items' list
wine_api = next(
    a for a in apis
    if 'api.danmurphys.com.au/apis/ui/ProductGroup/Products/wine%20all' in a['url']
    and isinstance(a.get('data'), dict)
    and 'Items' in a['data']
)
print(f"API endpoint : {wine_api['url']}")
print(f"Response keys: {list(wine_api['data'].keys())}")
print(f"Total available: {wine_api['data'].get('TotalRecordCount', '?')} products")
_items = wine_api['data']['Items']
_sample = next(iter(_items.values() if isinstance(_items, dict) else _items))
print(f"Fields per item: {list(_sample.keys())}")

# Step 2: paginate the API directly — 5 pages × 24 = 120 wines, no browser required
wines = paginate_api(
    wine_api['url'],
    payload={
        'pageSize': 5, 'pageNumber': 1,
        'sortType': 'Relevance', 'Location': 'ProductGroup',
        'Filters': [], 'ShowOnlyAvailable': False,
    },
    page_field='pageNumber',
    size_field='pageSize',
    results_field='Items',
    max_pages=1,
	save=True,
    save_file='test_page_{page}.json',
    verify=False,
)
print(f"\nCollected {len(wines)} wines")

# Step 3: save full product data — includes Price, PromoPrice, Name, Brand, Rating, etc.
Path('wines.json').write_text(json.dumps(wines, indent=2))
print(f"Saved to wines.json ({Path('wines.json').stat().st_size // 1024} KB)")

# preview first result
wines[0]

API endpoint : https://api.danmurphys.com.au/apis/ui/ProductGroup/Products/wine%20all
Response keys: ['Aggregations', 'Banners', 'Cards', 'DisplayName', 'SearchSource', 'Items', 'TotalRecordCount']
Total available: 7837 products
Fields per item: ['Name', 'PackDefaultStockCode', 'PackParentStockCode', 'Products', 'PackMessage', 'IsInDefaultList', 'IsPersonalised']
Page 1 already saved, skipping fetch

Collected 7 wines
Saved to wines.json (0 KB)


'Aggregations'

In [ ]:
#| eval: false
wines=paginate_api(wine_api['url'], page_field='pageNumber', size_field='pageSize', results_field='Items',
                   payload={'pageSize': 48, 'pageNumber': 1, 'sortType': 'Relevance', 'Location': 'ProductGroup',
		'Filters': [], 'ShowOnlyAvailable': False}, max_pages=100, save=True,
                   save_file='downloads/danmurphys_wines_page_{page}.json', verify=False)
Path('wines.json').write_text(json.dumps(wines, indent=2))
print(f"Saved to wines.json ({Path('wines.json').stat().st_size // 1024} KB)")

Page 1 already saved, skipping fetch
Page 2 already saved, skipping fetch
Page 3 already saved, skipping fetch
Page 4 already saved, skipping fetch
Page 5 already saved, skipping fetch
Page 6 already saved, skipping fetch
Page 7 already saved, skipping fetch
Page 8 already saved, skipping fetch
Page 9 already saved, skipping fetch
Page 10 already saved, skipping fetch
Page 11 already saved, skipping fetch
Page 12 already saved, skipping fetch
Page 13 already saved, skipping fetch
Page 14 already saved, skipping fetch
Page 15 already saved, skipping fetch
Page 16 already saved, skipping fetch
Page 17 already saved, skipping fetch
Page 18 already saved, skipping fetch
Page 19 already saved, skipping fetch
Page 20 already saved, skipping fetch
Page 21 already saved, skipping fetch
Page 22 already saved, skipping fetch
Page 23 already saved, skipping fetch
Page 24 already saved, skipping fetch
Page 25 already saved, skipping fetch
Page 26 already saved, skipping fetch
Page 27 already saved

In [ ]:
#| eval: false
L(apis).filter(lambda a: 'api.danmurphys.com.au/apis/ui/ProductGroup/Products/wine%20all' in a['url'])[0]['url']

'https://api.danmurphys.com.au/apis/ui/ProductGroup/Products/wine%20all'

In [ ]:
# paginate_api: test with JSONPlaceholder (free public REST API, GET-based)
posts = paginate_api(
    'https://jsonplaceholder.typicode.com/posts',
    payload={'_page': 1, '_limit': 5},
    page_field='_page',
    size_field='_limit',
    method='GET',
    verify=False,
)
assert len(posts) >= 5
assert 'title' in posts[0]

[2026-06-03 08:27:32] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=1&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:33] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=2&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:33] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=3&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:34] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=4&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:35] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=5&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:36] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=6&_limit=5> (referer: https://www.google.com/)
[2026-06-03 08:27:36] INFO: Fetched (200) <GET https://jsonplaceholder.typicode.com/posts?_page=7&_limit=5> (referer: https://www.google.com/)

## YouTube

`search_yt()` runs a YouTube search and returns metadata for each video. `read_yt()` fetches the auto-generated English captions as plain text, disk-cached by video ID. `download_yt()` saves audio or video to disk.

In [ ]:
#| export
_yt_cache = Index(str(save_path('yt_cache.db')))

def _yt_id(url:str) -> str:
    "Extract 11-char video ID from any YouTube URL or pass through bare ID"
    if m := re.search(r'(?:v=|youtu\.be/|/shorts/|/embed/)([A-Za-z0-9_-]{11})(?![A-Za-z0-9_-])', url): return m.group(1)
    return url

def _parse_vtt(vtt:str) -> str:
    "Strip VTT timing lines and cue tags, deduplicate adjacent lines, return plain text"
    lines = []
    for line in vtt.splitlines():
        if (re.match(r'^\d{2}:\d{2}', line) or '-->' in line
                or re.match(r'^(WEBVTT|Kind:|Language:|NOTE)', line)
                or not line.strip()): continue
        text = re.sub(r'<[^>]+>', '', line).strip()
        if text and (not lines or lines[-1] != text): lines.append(text)
    return ' '.join(lines)

In [ ]:
#| export
def search_yt(q:str, n:int=10) -> L:
    "Search YouTube; returns L of dicts: id, title, url, duration, view_count, channel, description, thumbnail"
    try:
        with yt_dlp.YoutubeDL({'quiet': True, 'extract_flat': True}) as ydl:
            info = ydl.extract_info(f'ytsearch{n}:{q}', download=False)
        return L({'id': e.get('id'),
                  'title': e.get('title'),
                  'url': e.get('url') or f'https://www.youtube.com/watch?v={e.get("id")}',
                  'duration': e.get('duration'),
                  'view_count': e.get('view_count'),
                  'channel': e.get('channel'),
                  'description': e.get('description'),
                  'thumbnail': next((t['url'] for t in reversed(e.get('thumbnails', [])) if 'url' in t), None)}
                 for e in info.get('entries', []))
    except Exception as e:
        print(f'search_yt error: {e}')
        return L()

In [ ]:
results = search_yt('3blue1brown neural networks', n=3)
assert isinstance(results, L), f"expected L, got {type(results)}"
assert len(results) >= 1, f"expected results, got {len(results)}"
assert any(kw in results[0]['title'].lower() for kw in ('3blue1brown', 'neural', 'network')), \
    f"unexpected title: {results[0]['title']}"
assert results[0]['url'].startswith('https://www.youtube.com'), f"bad url: {results[0]['url']}"
print(results[0]['title'], '|', results[0]['url'])

But what is a neural network? | Deep learning chapter 1 | https://www.youtube.com/watch?v=aircAruvnKk


In [ ]:
#| export
def read_yt(url:str, force:bool=False) -> dict:
    "Fetch YouTube metadata + English transcript (auto-captions); result disk-cached by video ID"
    vid = _yt_id(url)
    if not force and vid in _yt_cache: return _yt_cache[vid]
    with yt_dlp.YoutubeDL({'quiet': True, 'skip_download': True}) as ydl:
        info = ydl.extract_info(f'https://www.youtube.com/watch?v={vid}', download=False)
    caps = info.get('automatic_captions', {})
    en_caps = caps.get('en') or caps.get('en-orig') or []
    vtt_url = next((c['url'] for c in en_caps if c.get('ext') == 'vtt'), None)
    source = ''
    if vtt_url:
        r = http_get(vtt_url)
        if r.status_code == 200: source = _parse_vtt(r.text)
    res = dict(title=info.get('title', ''),
               channel=info.get('channel') or info.get('uploader', ''),
               description=info.get('description', ''),
               duration=info.get('duration'),
               upload_date=info.get('upload_date'),
               source=source)
    _yt_cache[vid] = res
    return res

In [ ]:
meta = read_yt('https://www.youtube.com/watch?v=aircAruvnKk')
assert meta['title'], "title should be non-empty"
assert isinstance(meta['source'], str), "source should be a string"
assert len(meta['source']) > 100, f"transcript too short: {len(meta['source'])} chars"
assert '3blue1brown' in meta['channel'].lower(), f"unexpected channel: {meta['channel']}"
print(f"title: {meta['title']}")
print(f"transcript preview: {meta['source'][:200]}")

title: But what is a neural network? | Deep learning chapter 1
transcript preview: [Music] This is a three. It's sloppily written and rendered at an extremely low resolution of 28x 28 pixels. But your brain has no trouble recognizing it as a three. And I want you to take a moment to


In [ ]:
#| export
def download_yt(url:str, format:str='audio', save_dir:str='.', quality:str=None) -> Path:
    "Download YouTube media; format='audio'|'video'|yt-dlp format string. Returns Path to saved file."
    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)
    outtmpl = str(save_dir / '%(title)s.%(ext)s')
    if format == 'audio':
        opts = {'quiet': True, 'format': 'bestaudio/best', 'outtmpl': outtmpl,
                'postprocessors': [{'key': 'FFmpegExtractAudio', 'preferredcodec': 'mp3',
                                    'preferredquality': quality or '192'}]}
    elif format == 'video':
        opts = {'quiet': True, 'format': 'bestvideo+bestaudio/best', 'outtmpl': outtmpl,
                'merge_output_format': 'mp4'}
    else:
        opts = {'quiet': True, 'format': format, 'outtmpl': outtmpl}
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
        stem = Path(ydl.prepare_filename(info)).stem
    ext = 'mp3' if format == 'audio' else ('mp4' if format == 'video' else '*')
    candidates = sorted(save_dir.glob(f'{stem}*.{ext}'))
    return candidates[0] if candidates else save_dir / f'{stem}.{ext}'

In [ ]:
#| eval: false
p = download_yt('https://www.youtube.com/watch?v=aircAruvnKk', format='audio', save_dir='/tmp/fossick_test')
assert p.exists(), f"file not found: {p}"
assert p.suffix == '.mp3', f"expected .mp3, got {p.suffix}"
print(f"saved to: {p} ({p.stat().st_size // 1024} KB)")

   saved to: /tmp/fossick_test/But what is a neural network？ ｜ Deep learning chapter 1.mp3 (26250 KB)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## Install

In [ ]:
#| export
def repo_root() -> Path:
    'Find the root of the current git repository, or None if not in a repo.'
    return first((Path.cwd(), *Path.cwd().parents), lambda p: (p/'.git').exists())

def mv_skill_md(dry_run=True, dir=None) -> None:
    'Copy bundled SKILL.md to skill directories.'
    base = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    if not (src := base.joinpath('SKILL.md')).exists(): return
    root = dir or repo_root() or '.'
    ts = [Path(root)/'.agents/skills/fossick/SKILL.md', Path('.claude/skills/fossick/SKILL.md')]
    if dry_run: print(f'Would copy {src} to: {list(map(str,ts))}')
    else:
        [p.mk_write(src.read_text(encoding='utf-8')) for p in ts]
        print(f'Installed → {list(map(str,ts))}')

In [ ]:
root = repo_root()
assert root is not None and (root/'.git').exists(), f"Expected git root, got {root}"
mv_skill_md(dry_run=True)